<a href="https://colab.research.google.com/github/vineetsalar88/ResearchPaper2/blob/master/March2026/11MarchResnet18.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

train_dir = "/content/drive/MyDrive/ResearchData/DatasetTrainVsTest10march/Train"
val_dir = "/content/drive/MyDrive/ResearchData/DatasetTrainVsTest10march/Val"

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5])
])

train_dataset = ImageFolder(train_dir, transform=transform)
val_dataset = ImageFolder(val_dir, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print(train_dataset.classes)

['0', '1', '2', '3', '4', '5']


In [ ]:
transforms.RandomRotation(20)
transforms.RandomHorizontalFlip()
transforms.ColorJitter()

ColorJitter(brightness=None, contrast=None, saturation=None, hue=None)

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.resnet18(pretrained=True)

# Freeze backbone
for param in model.parameters():
    param.requires_grad = False

# Replace final layer
num_classes = 6   # change according to your dataset
model.fc = nn.Linear(model.fc.in_features, num_classes)

model = model.to(device)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 198MB/s]


In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

In [ ]:
epochs = 15

train_acc_history = []
val_acc_history = []

for epoch in range(epochs):

    # -------- TRAIN --------
    model.train()

    correct = 0
    total = 0
    train_loss = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_acc = 100 * correct / total
    train_acc_history.append(train_acc)

    # -------- VALIDATION --------
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_acc = 100 * correct / total
    val_acc_history.append(val_acc)

    print(f"Epoch [{epoch+1}/{epochs}] "
          f"Train Acc: {train_acc:.2f}% "
          f"Val Acc: {val_acc:.2f}%")

Epoch [1/15] Train Acc: 73.55% Val Acc: 55.87%
Epoch [2/15] Train Acc: 73.55% Val Acc: 54.09%
Epoch [3/15] Train Acc: 75.23% Val Acc: 54.09%
Epoch [4/15] Train Acc: 74.77% Val Acc: 54.80%
Epoch [5/15] Train Acc: 77.83% Val Acc: 55.52%
Epoch [6/15] Train Acc: 74.01% Val Acc: 54.45%
Epoch [7/15] Train Acc: 76.91% Val Acc: 53.38%
Epoch [8/15] Train Acc: 78.13% Val Acc: 57.30%
Epoch [9/15] Train Acc: 78.29% Val Acc: 56.23%
Epoch [10/15] Train Acc: 76.91% Val Acc: 52.67%
Epoch [11/15] Train Acc: 75.08% Val Acc: 54.80%
Epoch [12/15] Train Acc: 80.12% Val Acc: 57.30%
Epoch [13/15] Train Acc: 80.43% Val Acc: 58.01%
Epoch [14/15] Train Acc: 78.90% Val Acc: 58.01%
Epoch [15/15] Train Acc: 81.19% Val Acc: 57.65%
